# Lineups
Welcome to Lineups, a new fantasy baseball game! Unlike traditional fantasy baseball games where points are alloted to players for the stats they accumulate, in Lineups, you will draft your players and then set a lineup to play an actual baseball game simulated using your players actual performance in their real-life games. Instead of being awarded points for stats, you will play manager and try to put your team in the best position to manufacture as many runs as they can. It's your fantasy team playing real baseball!

Note: For this demo, game data is generated from a previous date in September 2023
## Load Data

To begin, hover over the line below and click on the black triangle, then move to the next section.

In [ ]:
import json
import pandas as pd

with open('players.json', 'r') as f:
    players = json.load(f)
    
with open('pbp_data.json', 'r') as f:
    pbp_data = json.load(f)   
    
# Adjusting the provided code to loop through all games in pbp_data
data_for_all_games = []

for game_id in pbp_data.keys():
    all_innings = pbp_data[game_id]['game']['innings']
    for inning in all_innings:
        for half_data in inning.get('halfs', []):
            for event in half_data.get('events', []):
                at_bat = event.get('at_bat', {})
                batter_id = at_bat.get('hitter_id')
                sequence_number = at_bat.get('sequence_number')
                outcome_id = at_bat.get('events', [{}])[-1].get('outcome_id')
                data_for_all_games.append([game_id, batter_id, sequence_number, outcome_id])

# Creating the DataFrame for all games
df_at_bats_all_games = pd.DataFrame(data_for_all_games, columns=['game_id', 'batter_id', 'sequence_number', 'outcome_id'])
df_at_bats_all_games = df_at_bats_all_games[df_at_bats_all_games['outcome_id'].notna()]
df_at_bats_all_games.to_csv('plays_all_games.csv', index=False)



## Select Players for Your Team

Select a starting player for each position. The players listed in the dropdowns are players eligible for that position. If a player is listed, it means his team has a game scheduled on the date the data was pulled from, but it does not account for injured list status. After selecting a player for each position, you will select six bench players of any position who will be available to pinch-hit in case any of your starters run out of at bats before the game ends.

(Only batting outcomes are used for this game, fileding, pitching, and baserunning stats are ignored)

#### Click the black triangle below to load the team selection menu

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Define the 'selected_players' dictionary
selected_players = {
    'C': {'id': '', 'name': '', 'lineup_slot': ''},
    '1B': {'id': '', 'name': '', 'lineup_slot': ''},
    '2B': {'id': '', 'name': '', 'lineup_slot': ''},
    'SS': {'id': '', 'name': '', 'lineup_slot': ''},
    '3B': {'id': '', 'name': '', 'lineup_slot': ''},
    'OF': [{'id': '', 'name': '', 'lineup_slot': ''}, {'id': '', 'name': '', 'lineup_slot': ''}, {'id': '', 'name': '', 'lineup_slot': ''}],
    'DH': {'id': '', 'name': '', 'lineup_slot': ''},
    'PH': [{'id': '', 'name': '', 'lineup_slot': '', 'bench_order': ''}, {'id': '', 'name': '', 'lineup_slot': '', 'bench_order': ''}, {'id': '', 'name': '', 'lineup_slot': '', 'bench_order': ''}, {'id': '', 'name': '', 'lineup_slot': '', 'bench_order': ''}, {'id': '', 'name': '', 'lineup_slot': '', 'bench_order': ''}, {'id': '', 'name': '', 'lineup_slot': '', 'bench_order': ''}]
}

# Create a list of player names for each position
player_names_by_position = {
    'C': ['Select Player'] + [player_info['name'] for player_info in players.values() if player_info['position'] == 'C'],
    '1B': ['Select Player'] + [player_info['name'] for player_info in players.values() if player_info['position'] == '1B'],
    '2B': ['Select Player'] + [player_info['name'] for player_info in players.values() if player_info['position'] == '2B'],
    '3B': ['Select Player'] + [player_info['name'] for player_info in players.values() if player_info['position'] == '3B'],
    'SS': ['Select Player'] + [player_info['name'] for player_info in players.values() if player_info['position'] == 'SS'],
    'OF': ['Select Player'] + [player_info['name'] for player_info in players.values() if player_info['position'] == 'OF'],
    'DH': ['Select Player'] + [player_info['name'] for player_info in players.values() if player_info['position'] == 'DH'],
    'PH': ['Select Player'] + [player_info['name'] for player_info in players.values() if player_info['position'] != 'P']
}

# Create a list of positions
positions = ['C', '1B', '2B', '3B', 'SS', 'OF', 'OF', 'OF', 'DH', 'PH', 'PH', 'PH', 'PH', 'PH', 'PH']

# Create a list of dropdown menus
dropdowns = [widgets.Dropdown(options=player_names_by_position[pos], description=pos) for pos in positions]

# Create a button
button = widgets.Button(description='Select team')

# Create an output widget to display the selected players
output = widgets.Output()

# Define a function to run when the button is clicked
def on_button_clicked(b):
    # Clear the output
    output.clear_output()

    # Get the player names from the dropdown menus
    player_names = [dropdown.value for dropdown in dropdowns]

    # Check if all positions are filled
    if 'Select Player' in player_names:
        with output:
            print('Error: All positions must be filled.')
        return

    # Check if there are any duplicate players
    if len(player_names) != len(set(player_names)):
        with output:
            print('Error: Each player can only be selected once.')
        return

    for i, pos in enumerate(positions):
        # Get the player's id
        matching_ids = [id for id, player_info in players.items() if player_info['name'] == player_names[i]]
    
        if not matching_ids:
            print(f"No match found for player name: {player_names[i]}")
            continue
        player_id = matching_ids[0]

        if pos in ['OF', 'PH']:
            # Find the first dictionary in the list that has an empty 'name' field
            for player_info in selected_players[pos]:
                if player_info['name'] == '':
                    player_info.update({'id': player_id, 'name': player_names[i], 'lineup_slot': ''})
                    if pos == 'PH':
                        player_info.update({'position': 'PH', 'bench_order': ''})
                    break
        else:
            selected_players[pos] = {'id': player_id, 'name': player_names[i], 'lineup_slot': ''}

    # Display the 'selected_players' dictionary
    with output:
        print(selected_players)

# Set the function to run when the button is clicked
button.on_click(on_button_clicked)

# Display the widgets
display(*dropdowns, button, output)


## Set Your Lineup

Now you will prove your moxy by setting your lineup. Just like in a real game, the order of events will determine how many runs your team scores, so even someone with the exact same team may have a completely different outcome in the game depending on how the lineup is set. What ever your player does during their at-bats in their real-life game will be exactly what they do in this fantasy game, so the order matters!

If a player runs out of at-bats before the game is over, pinch-hitters will be subbed in according to the order you set here for your pinch-hitters (PH1 will be first off the bench, PH2 will be second, etc.)

#### Click the black triangle below to load the lineup menu

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Create a list of player names for each position excluding PH
player_names_excluding_PH = ['Select Player'] 
for pos, player_info in selected_players.items():
    if pos != 'PH':
        if pos == 'OF':
            for player in player_info:
                if player['name']:  # Check if the name field is not empty
                    player_names_excluding_PH.append(player['name'])
        else:
            if player_info['name']:  # Check if the name field is not empty
                player_names_excluding_PH.append(player_info['name'])

# Create a list of player names for PH
player_names_PH = ['Select Player'] + [player_info['name'] for player_info in selected_players['PH'] if isinstance(player_info, dict)]

# Create a list of lineup slots
lineup_slots = [str(i) for i in range(1, 10)]

# Create a list of PH slots
ph_slots = ['PH' + str(i) for i in range(1, 7)]

# Create a list of dropdown menus for lineup slots
dropdowns_lineup = [widgets.Dropdown(options=player_names_excluding_PH, description=slot) for slot in lineup_slots]

# Create a list of dropdown menus for PH slots
dropdowns_ph = [widgets.Dropdown(options=player_names_PH, description=slot) for slot in ph_slots]

# Create a button
button = widgets.Button(description='Set lineup')

# Create an output widget to display the lineup
output = widgets.Output()

# Create function to run on button click
def on_button_clicked(b):
    # Clear the output
    output.clear_output()

    # Get the player names from the dropdown menus
    player_names_lineup = [dropdown.value for dropdown in dropdowns_lineup]
    player_names_ph = [dropdown.value for dropdown in dropdowns_ph]

    # Check if all lineup slots are filled
    if 'Select Player' in player_names_lineup:
        with output:
            print('Error: All lineup slots must be filled.')
        return

    # Check if all PH slots are filled
    if 'Select Player' in player_names_ph:
        with output:
            print('Error: All PH slots must be filled.')
        return

    # Check if there are any duplicate players
    if len(player_names_lineup + player_names_ph) != len(set(player_names_lineup + player_names_ph)):
        with output:
            print('Error: Each player can only be selected once.')
        return

    # Set the lineup slots for the players in the 'selected_players' dictionary
    for i, slot in enumerate(lineup_slots):
        for pos, player_info in selected_players.items():
            if isinstance(player_info, list):
                for player in player_info:
                    if player['name'] == player_names_lineup[i]:
                        player['lineup_slot'] = int(slot)
            else:
                if player_info['name'] == player_names_lineup[i]:
                    player_info['lineup_slot'] = int(slot)

    # Set the bench order for the PHs in the 'selected_players' dictionary
    for i, player_name in enumerate(player_names_ph):
        for player in selected_players['PH']:
            if player['name'] == player_name:
                player['bench_order'] = i + 1
                break

    # Display the 'selected_players' dictionary
    with output:
        print(selected_players)

# Set the function to run when the button is clicked
button.on_click(on_button_clicked)

# Display the widgets
display(*dropdowns_lineup, widgets.HTML('<br>'), *dropdowns_ph, button, output)

## Gameplay Simulation

Now we see how well you did as a manager. The game will be simulated using the players at-bats results from their games, according to the batting order you set. In the rules of this simulation: a runner on second base will always score on a hit; a runner on third will tag and score on a flyout; a player who grounds out while a force is in play will always trigger a double play. The game ends after nine innings, or when no players have any remaining at-bats.

#### Click the black traingle below to run the simulation

In [ ]:
from collections import deque

# Load the at_bat_codebook file to interpret the outcome_id
codebook = pd.read_csv('at_bat_codebook.csv')

# Load the plays_all_games file to match the player id
plays_all_games = pd.read_csv('plays_all_games.csv')

lineup = selected_players

# Extract players and their information from the loaded JSON file
lineup_list = []
for key, value in lineup.items():
    if isinstance(value, list):
        for player_info in value:
            if 'lineup_slot' in player_info and player_info['lineup_slot']:
                lineup_list.append(player_info)
    else:
        lineup_list.append(value)

# Sort the lineup_list based on lineup_slot
lineup_list = sorted(lineup_list, key=lambda x: x['lineup_slot'])

# Define class object for simulation:
out_messages = {
    'oFC': "grounds out.",
    'oFO': "flies out.",
    'oGO': "grounds out.",
    'oLO': "lines out.",
    'oPO': "pops out.",
    'kKL': "strikes out looking.",
    'kKS': "strikes out swinging.",
    'kFT': "strikes out swinging."
    }

class FantasyGame:
    def __init__(self):
        self.inning = 1
        self.outs = 0
        self.bases = {'1B': None, '2B': None, '3B': None}
        self.score = 0
        self.current_batter = None
        self.game_over = False
        

    def hit(self, bases_advanced):
        if bases_advanced == 1:  # This is a single
            # Handle runner on 2B scoring on a single
            if self.bases['2B']:
                print(f'{self.bases["2B"]["name"]} scores from second on the single.')
                self.bases['2B'] = None
                self.score += 1

        # The logic for handling other runners remains the same
        runs_scored = []
        for base in reversed(range(1, 4)):
            runner = self.bases[f'{base}B']
            if runner is not None:
                new_base = base + bases_advanced
                if new_base > 3:
                    self.score += 1
                    runs_scored.append(runner['name'])
                else:
                    self.bases[f'{new_base}B'] = runner
                self.bases[f'{base}B'] = None
        new_base = bases_advanced
        if new_base > 3:
            self.score += 1
            runs_scored.append(self.current_batter['name'])
        else:
            self.bases[f'{new_base}B'] = self.current_batter
        if runs_scored:
            print(', '.join(runs_scored) + ' scores.')


        
    def walk(self):
        # Scenario: {1B: Runner A, 2B: Runner B, 3B: Runner C}
        # Result: {1B: batter, 2B: Runner A, 3B: Runner B, Runner C scores}
        if self.bases['1B'] and self.bases['2B'] and self.bases['3B']:
            self.score += 1
            print(f'{self.bases["3B"]["name"]} scores.')
            self.bases['3B'] = self.bases['2B']
            self.bases['2B'] = self.bases['1B']
        
    
        # Scenario: {1B: Runner A, 2B: Runner B, 3B: Empty}
        # Result: {1B: batter, 2B: Runner A, 3B: Runner B}
        elif self.bases['1B'] and self.bases['2B']:
            self.bases['3B'] = self.bases['2B']
            self.bases['2B'] = self.bases['1B']

        # Scenario: {1B: Runner A, 2B: Empty, 3B: Runner B}
        # Result: {1B: batter, 2B: Runner A, 3B: Runner B}
        elif self.bases['1B'] and self.bases['3B']:
            self.bases['2B'] = self.bases['1B']

        # Scenario: {1B: Runner A, 2B: Empty, 3B: Empty}
        # Result: {1B: batter, 2B: Runner A, 3B: Empty}
        elif self.bases['1B']:
            self.bases['2B'] = self.bases['1B']

        # The batter always takes 1B
        self.bases['1B'] = self.current_batter

        
    def out(self):
        self.outs += 1  # Increment the outs
    
        if self.outs >= 3:  # Check if outs has reached 3
            self.print_state()  # Print the game state with Outs: 3
        
            # Reset the outs and increment the inning
            self.outs = 0
            self.inning += 1
            # Clear the bases
            self.bases = {'1B': None, '2B': None, '3B': None}

            # Check if the game should end
            if self.inning > 9:
                self.game_over = True
                
                
    def double_play(self):
        # Handle the scenario when bases are loaded
        if self.bases['1B'] and self.bases['2B'] and self.bases['3B']:
            print(f'grounds into double play. {self.bases["3B"]["name"]} out at home.')
            self.bases['3B'] = self.bases['2B']
            self.bases['2B'] = self.bases['1B']
            self.bases['1B'] = None
            self.out()
            self.out()
            return

        # For other scenarios, handle the batter and runner at 1B
        print(f'grounds into double play. {self.bases["1B"]["name"]} out at 2B.')
        self.bases['1B'] = None
        self.out()
        self.out()

        # Handle additional runners
        if self.bases['2B']:
            # Runners on 1B and 2B scenario
            self.bases['3B'] = self.bases['2B']
            self.bases['2B'] = None
        # Note: For the runner on 1B and 3B scenario, the runner at 3B just stays put.


            
    def print_state(self):
        print(f'Inning: {self.inning}')
        print(f'Outs: {self.outs}')
        print(f'Bases: {self.bases}')
        print(f'Score: {self.score}')
        print('='*40)
        
    def process_at_bat(self, outcome_id):
        print(f"{self.current_batter['name']} ", end='')
        if outcome_id.startswith('aS'):
            print(f'singles. ({outcome_id})')
            self.hit(1)
        elif outcome_id.startswith('aD'):
            print(f'doubles. ({outcome_id})')
            self.hit(2)
        elif outcome_id.startswith('aT'):
            print(f'triples. ({outcome_id})')
            self.hit(3)
        elif outcome_id.startswith('aHR'):
            print(f'homers. ({outcome_id})')
            self.hit(4)
        elif outcome_id in ['aBB', 'aIBB', 'bB']:
            print(f'walks. ({outcome_id})')
            self.walk()
        elif outcome_id.startswith('o') or outcome_id.startswith('k'):
            # Check if double play conditions are met
            if self.outs < 2 and self.bases['1B'] and outcome_id in ['oGO', 'oFC']:
                self.double_play()
            else:
                # Check for fly out with runner on 3B and less than 2 outs
                if self.bases['3B'] and self.outs < 2 and outcome_id == 'oFO':
                    print(f'flies out, {self.bases["3B"]["name"]} tags up and scores. ({outcome_id})')
                    self.score += 1
                    self.bases['3B'] = None
                    self.out()
                else:
                    # Handle regular out
                    print(f'{out_messages[outcome_id]} ({outcome_id})')
                    self.out()
        self.print_state()

        
# Define simulation loop

# Step 1: Initial Setup (Re-initialize necessary components)

# Convert the lineup into a queue (deque), ordered by 'lineup_slot'
lineup_queue = deque()
for key, value in lineup.items():
    if isinstance(value, list):  # For keys like 'OF' and 'PH'
        for player_info in value:
            if 'lineup_slot' in player_info and player_info['lineup_slot']:  # Only add if lineup_slot is present and not empty
                lineup_queue.append((key, player_info))
    else:
        lineup_queue.append((key, value))

# Sort the lineup_queue based on lineup_slot
lineup_queue = deque(sorted(lineup_queue, key=lambda x: x[1]['lineup_slot']))

# Convert the pinch hitters into a separate queue, sorted by bench_order
ph_queue = deque(sorted(lineup['PH'], key=lambda x: x['bench_order']))

# Initialize a dictionary to track the last at-bat sequence for each player
all_players = {player_info['id']: 0 for _, player_info in lineup_queue}
for ph in ph_queue:
    all_players[ph['id']] = 0

last_sequence_listed = all_players.copy()

game = FantasyGame()

while not game.game_over:
    # Check if the lineup queue is empty (no players left to bat)
    if not lineup_queue:
        print("Warning: No players left to bat. Ending the game.")
        break
    
    
    
    # Loop through the lineup
    for _ in range(len(lineup_queue)):
        position, player_info = lineup_queue.popleft()
        player_id = player_info['id']
        
            # --- New Check: End the game if game_over is True ---
        if game.game_over:
            break
        
        # Find the player's next at-bat data
        matching_plays = plays_all_games[
            (plays_all_games['batter_id'] == player_id) & 
            (plays_all_games['sequence_number'] > last_sequence_listed[player_id])
        ]
        
        # If the player has multiple games, consider only the first game_id
        if matching_plays['game_id'].nunique() > 1:
            first_game_id = matching_plays['game_id'].iloc[0]
            matching_plays = matching_plays[matching_plays['game_id'] == first_game_id]
        
        # If the player has more at-bats
        if not matching_plays.empty:
            all_listed = False
            at_bat = matching_plays.nsmallest(1, 'sequence_number').iloc[0]
            outcome_id = at_bat['outcome_id']
            game.current_batter = player_info
            game.process_at_bat(outcome_id)
            last_sequence_listed[player_id] = at_bat['sequence_number']
            lineup_queue.append((position, player_info))  # Re-append the player to the lineup queue
        
        # If the player is out of at-bats, check for available pinch hitters
        else:
            if ph_queue:  # Check if pinch hitters are available
                new_player_info = ph_queue.popleft()
                print(f"Subbing in {new_player_info['name']} for {player_info['name']} in position {position}")
                
                # Immediately get the at-bat for the subbed player
                new_player_id = new_player_info['id']
                new_matching_plays = plays_all_games[
                    (plays_all_games['batter_id'] == new_player_id) &
                    (plays_all_games['sequence_number'] > last_sequence_listed[new_player_id])
                ]
                
                # If the subbed player has multiple games, consider only the first game_id
                if new_matching_plays['game_id'].nunique() > 1:
                    first_game_id = new_matching_plays['game_id'].iloc[0]
                    new_matching_plays = new_matching_plays[new_matching_plays['game_id'] == first_game_id]
                
                if not new_matching_plays.empty:
                    all_listed = False
                    at_bat = new_matching_plays.nsmallest(1, 'sequence_number').iloc[0]
                    outcome_id = at_bat['outcome_id']
                    game.current_batter = new_player_info
                    game.process_at_bat(outcome_id)
                    last_sequence_listed[new_player_id] = at_bat['sequence_number']
                
                lineup_queue.append((position, new_player_info))  # Append the subbed player to the lineup queue
            else:
                print(f"{player_info['name']} is out of at-bats and no pinch hitters available for position {position}")
    
    # Check for end of game condition (after nine innings)
    if game.game_over:
        print("Game Over")
        break
    
    
print("Final Game State:")
game.print_state()

Thairo Estrada grounds out. (oGO)
Inning: 1
Outs: 1
Bases: {'1B': None, '2B': None, '3B': None}
Score: 0
Josh Rojas strikes out looking. (kKL)
Inning: 1
Outs: 2
Bases: {'1B': None, '2B': None, '3B': None}
Score: 0
Subbing in TJ Friedl for Luisangel Acuña in position SS
TJ Friedl grounds out. (oGO)
Inning: 1
Outs: 3
Bases: {'1B': None, '2B': None, '3B': None}
Score: 0
Inning: 2
Outs: 0
Bases: {'1B': None, '2B': None, '3B': None}
Score: 0
Subbing in Matt McLain for Jared Young in position 1B
Subbing in Cam Gallagher for Jorge Soler in position DH
Subbing in Mark Vientos for Trent Grisham in position OF
Mark Vientos lines out. (oLO)
Inning: 2
Outs: 1
Bases: {'1B': None, '2B': None, '3B': None}
Score: 0
Luke Maile grounds out. (oGO)
Inning: 2
Outs: 2
Bases: {'1B': None, '2B': None, '3B': None}
Score: 0
Subbing in Nick Castellanos for Brennen Davis in position OF
Nick Castellanos strikes out swinging. (kKS)
Inning: 2
Outs: 3
Bases: {'1B': None, '2B': None, '3B': None}
Score: 0
Inning: 3
Out

KeyError: 'oSF'